# Reproduce the ASR figures

Run this notebook to regenerate the paper's ASR figures and summary tables on CPU.
The included compact dataset contains candidate and success counts for each target
and bundle size. It contains no query text or model responses and requires no GPU
or API key. The same workflow is available as `python scripts/plot_results.py`.

**Definitions.** Total bundle size $K$ includes the designated target. At $K=1$
there is one direct request; at each $K=2,\ldots,6$ there are 20 composed candidates.
Cumulative ASR is the fraction of the 100 targets reached at least once through
size $K$. Incremental gain is the fraction first reached at that size.
For the generator comparison, both curves use each downstream model's Qwen3-32B
direct-request outcomes at $K=1$.


In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts/plot_results.py").is_file()
)
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

from plot_results import load_outcomes, reproduce_figures

INPUT_FILE = REPO_ROOT / "data/processed/target_outcomes.csv"
OUTPUT_DIR = REPO_ROOT / "outputs/figures"


## Inspect the included counts

Validation checks all ten conditions, a shared roster of 100 targets, sizes 1–6,
and the expected candidate counts. Source-file hashes and the aggregation rule
are recorded in `data/processed/provenance.json`.


In [ ]:
outcomes = load_outcomes(INPUT_FILE)
display(outcomes.head(12))
print(f"{len(outcomes):,} compact rows represent {outcomes.n_candidates.sum():,} candidate outcomes.")


## Generate the figures and tables

The plotting and metric functions live in `scripts/plot_results.py`, so the
notebook and command-line workflow use the same implementation.


In [ ]:
tables = reproduce_figures(INPUT_FILE, OUTPUT_DIR)
display(tables["table_cumulative_asr_32b.csv"])


In [ ]:
for filename in (
    "figure_A_incremental_qwen4b_comparison.png",
    "figures_BC_side_by_side_checkpoint_comparisons.png",
    "figure_D_matched_generator_comparison.png",
):
    display(Image(filename=str(OUTPUT_DIR / filename)))


## Optional: rebuild from raw results

If you have the original labeled result files, prepare their canonical paths using
`python scripts/prepare_data.py /path/to/Intent`, then run:

```bash
python scripts/plot_results.py --from-results data/results \
    --save-outcomes data/processed/target_outcomes.csv
```

The raw import validates the complete experimental design and records source-file
hashes. CSV files take precedence over archived Excel files with the same stem.
